In [0]:
%run ./99_Config

In [0]:
%run ./98_Utilities

In [0]:
%run ./97_Logger

In [0]:
from pyspark.sql.functions import *

pipeline_start("06_Fraud_Detection")

In [0]:
silver_path = f"{SILVER_PATH}/transactions"
fraud_path = f"{GOLD_PATH}/fraud_transactions"

df = read_delta(silver_path)

rows_read = record_count(df)

In [0]:
fraud_df = (
    df
    .withColumn(
        "fraud_reason",
        when(col("amount") >= 100000, "HIGH_VALUE_TRANSACTION")
        .when(col("transaction_category") == "HIGH", "HIGH_CATEGORY")
        .when(col("transaction_type") == "ATM_WITHDRAWAL", "HIGH_RISK_ATM")
    )
    .filter(col("fraud_reason").isNotNull())
    .withColumn("fraud_detected_timestamp", current_timestamp())
)

fraud_rows = record_count(fraud_df)

In [0]:
write_delta(fraud_df, fraud_path, "overwrite")

create_delta_table(
    "gold_fraud_transactions",
    fraud_path
)

In [0]:
audit(
    notebook="06_Fraud_Detection",
    dataset="transactions",
    rows_read=rows_read,
    rows_written=fraud_rows,
    rejected_rows=0,
    status="SUCCESS"
)

log_success(
    notebook="06_Fraud_Detection",
    dataset="transactions",
    rows_read=rows_read,
    rows_written=fraud_rows,
    execution_time="Completed"
)

display_summary(
    "Fraud Detection",
    rows_read,
    fraud_rows,
    0
)

In [0]:
print("="*80)
print("FRAUD DETECTION SUMMARY")
print("="*80)
print(f"Rows Analysed : {rows_read}")
print(f"Fraud Records : {fraud_rows}")
print("="*80)

pipeline_end("06_Fraud_Detection")
